# Runtime comparison: DFN simulation vs trained MLP surrogate

This notebook benchmarks the runtime of one PyBaMM DFN formation simulation against the inference runtime of the trained MLP physics-guided sequence-to-sequence surrogate. It is intended to support the practical-efficiency discussion in the FAIEMA paper.

Recommended use:
1. Run this notebook from the `notebooks/` folder of the project, or from the project root.
2. Verify that the paths point to the correct `data/processed`, `models_nonoverlap`, and `results_runtime` folders.
3. Use the printed runtime table in the manuscript.


In [ ]:
# If PyBaMM is not installed in your environment, uncomment the next line.
# %pip install "pybamm==25.10.2" -q

from pathlib import Path
import json
import time
import random
import platform

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    import pybamm
    PYBAMM_AVAILABLE = True
except ImportError:
    PYBAMM_AVAILABLE = False
    print("PyBaMM is not installed. Surrogate timing can still run, but DFN timing will be skipped.")

SEED = 63
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
if PYBAMM_AVAILABLE:
    print("PyBaMM:", pybamm.__version__)


## Project paths and experiment settings

This assumes the same file naming convention used in the training notebooks.


In [ ]:
# Project paths
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models_nonoverlap")
RESULTS_DIR = Path("../results_runtime")
RAW_DIR = Path("../data/raw")

# Fallback if run from project root instead of notebooks/
if not PROCESSED_DIR.exists():
    PROCESSED_DIR = Path("data/processed")
    MODELS_DIR = Path("models_nonoverlap")
    RESULTS_DIR = Path("results_runtime")
    RAW_DIR = Path("data/raw")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SET = "physics"
INPUT_LENGTH = 50
PREDICTION_HORIZON = 50
STRIDE = 5
TEST_STRIDE = 50
BATCH_SIZE = 256

package_name = f"seq2seq_{FEATURE_SET}_L{INPUT_LENGTH}_H{PREDICTION_HORIZON}_S{STRIDE}"
NPZ_PATH = PROCESSED_DIR / f"{package_name}.npz"
SCALER_PATH = PROCESSED_DIR / f"{package_name}_scaler.json"
META_PATH = PROCESSED_DIR / f"{package_name}_metadata.csv"

model_name = f"mlp_direct_{FEATURE_SET}_L{INPUT_LENGTH}_H{PREDICTION_HORIZON}_S{STRIDE}"
MODEL_PATH = MODELS_DIR / f"{model_name}_best.pt"

print("NPZ:", NPZ_PATH.resolve())
print("Scaler:", SCALER_PATH.resolve())
print("Metadata:", META_PATH.resolve())
print("Model:", MODEL_PATH.resolve())


## Load processed data and trained MLP surrogate

In [ ]:
data = np.load(NPZ_PATH)
X_test = data["X_test"]
Y_test = data["Y_test"]

with open(SCALER_PATH, "r") as f:
    scaler = json.load(f)

input_cols = scaler["input_cols"]
output_cols = scaler["output_cols"]
y_mean = np.array(scaler["y_mean"], dtype=np.float32)
y_std = np.array(scaler["y_std"], dtype=np.float32)

input_dim = X_test.shape[-1]
output_dim = Y_test.shape[-1]

print("Input columns:", input_cols)
print("Output columns:", output_cols)
print("X_test:", X_test.shape, "Y_test:", Y_test.shape)

class Seq2SeqDataset(Dataset):
    def __init__(self, X, Y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = None if Y is None else torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.Y is None:
            return self.X[idx]
        return self.X[idx], self.Y[idx]

class MLPDirectSeq2Seq(nn.Module):
    def __init__(self, input_length, input_dim, hidden_dim, output_dim, horizon, dropout=0.2):
        super().__init__()
        self.horizon = horizon
        self.output_dim = output_dim
        self.network = nn.Sequential(
            nn.Linear(input_length * input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, horizon * output_dim),
        )

    def forward(self, x):
        batch_size = x.size(0)
        x = x.reshape(batch_size, -1)
        out = self.network(x)
        return out.view(batch_size, self.horizon, self.output_dim)

model = MLPDirectSeq2Seq(
    input_length=INPUT_LENGTH,
    input_dim=input_dim,
    hidden_dim=256,
    output_dim=output_dim,
    horizon=PREDICTION_HORIZON,
    dropout=0.2,
).to(device)

checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded model from epoch:", checkpoint.get("epoch", "unknown"))


## Select non-overlapping test windows

The original test windows were generated with stride 5. Since the prediction horizon is 50, selecting every 10th test window gives approximately non-overlapping forecast blocks.


In [ ]:
if TEST_STRIDE % STRIDE != 0:
    raise ValueError("TEST_STRIDE must be divisible by STRIDE.")

selection_factor = TEST_STRIDE // STRIDE

# All non-overlapping test windows
X_test_blocks_all = X_test[::selection_factor]
Y_test_blocks_all = Y_test[::selection_factor]

print("Original overlapping test windows:", X_test.shape[0])
print("Non-overlapping test windows, all test protocols:", X_test_blocks_all.shape[0])
print("Selection factor:", selection_factor)

# Select non-overlapping windows for each test protocol using metadata.
# This will be used to compare surrogate runtime with each DFN protocol runtime.
protocols_to_benchmark = ["multi_6", "slow_6", "fast_6"]
X_test_blocks_by_protocol = {}
Y_test_blocks_by_protocol = {}

if META_PATH.exists():
    meta_df = pd.read_csv(META_PATH)
    test_meta = meta_df[meta_df["split"] == "test"].reset_index(drop=True)

    for protocol in protocols_to_benchmark:
        overlap_idx = np.where(test_meta["dataset"].values == protocol)[0]
        nonoverlap_idx = overlap_idx[::selection_factor]

        X_test_blocks_by_protocol[protocol] = X_test[nonoverlap_idx]
        Y_test_blocks_by_protocol[protocol] = Y_test[nonoverlap_idx]

        print(f"Non-overlapping {protocol} windows:", X_test_blocks_by_protocol[protocol].shape[0])
else:
    print("Metadata file not found; protocol-wise surrogate timing will be skipped.")


## Benchmark surrogate inference runtime

In [ ]:
def sync_device():
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.synchronize()

@torch.no_grad()
def time_surrogate_inference(X_blocks, repeats=10, batch_size=256):
    loader = DataLoader(Seq2SeqDataset(X_blocks), batch_size=batch_size, shuffle=False)

    # Warm-up run. Important for GPU/MPS timing.
    for X_batch in loader:
        X_batch = X_batch.to(device)
        _ = model(X_batch)
    sync_device()

    times = []
    n_predictions = len(X_blocks)

    for _ in range(repeats):
        start = time.perf_counter()
        for X_batch in loader:
            X_batch = X_batch.to(device)
            _ = model(X_batch)
        sync_device()
        end = time.perf_counter()
        times.append(end - start)

    return {
        "n_windows": int(n_predictions),
        "mean_s": float(np.mean(times)),
        "std_s": float(np.std(times)),
        "min_s": float(np.min(times)),
        "max_s": float(np.max(times)),
        "repeats": repeats,
    }

# Runtime for all non-overlapping test windows together
surrogate_all = time_surrogate_inference(X_test_blocks_all, repeats=10, batch_size=BATCH_SIZE)
print("Surrogate timing, all test protocols:", surrogate_all)

# Runtime for each test protocol separately
surrogate_by_protocol = {}

for protocol, X_blocks in X_test_blocks_by_protocol.items():
    if X_blocks is not None and len(X_blocks) > 0:
        surrogate_by_protocol[protocol] = time_surrogate_inference(
            X_blocks,
            repeats=10,
            batch_size=BATCH_SIZE
        )
        print(f"Surrogate timing, {protocol}:", surrogate_by_protocol[protocol])
    else:
        print(f"No windows found for {protocol}.")


## Benchmark DFN formation simulations

This cell times PyBaMM DFN solves for `multi_6`, `slow_6`, and `fast_6` using the same model options, parameter set, and solver configuration as the dataset-generation notebook. The runtime includes solving the DFN model. A separate post-processing timing is also reported for evaluating variables on a 1-second grid and computing derived variables.

In [ ]:
dfn_results = {}

if PYBAMM_AVAILABLE:
    def build_dfn_model():
        return pybamm.lithium_ion.DFN(
            options={
                "open-circuit potential": "single",
                "intercalation kinetics": "symmetric Butler-Volmer",
                "thermal": "lumped",
                "surface form": "algebraic",
                "surface temperature": "lumped",
                "SEI": "ec reaction limited",
                "SEI film resistance": "distributed",
                "SEI porosity change": "true",
                "calculate discharge energy": "false",
                "cell geometry": "arbitrary",
                "calculate heat source for isothermal models": "false",
                "current collector": "uniform",
                "diffusivity": "single",
                "exchange-current density": "single",
                "lithium plating": "irreversible",
                "lithium plating porosity change": "true",
            }
        )

    parameter_values = pybamm.ParameterValues("Mohtat2020")
    parameter_values.update(
        {
            "Casing heat capacity [J.K-1]": 300.0,
            "Environment thermal resistance [K.W-1]": 2.0,
        },
        check_already_exists=False,
    )

    solver = pybamm.IDAKLUSolver(
        rtol=1e-4,
        atol=1e-6,
        options={"max_num_steps": 50000},
    )

    experiments_to_benchmark = {
        "multi_6": pybamm.Experiment([
            (
                "Rest for 45 minutes",
                "Charge at 0.045C until 2.4V",
                "Rest for 300 minutes",
                "Charge at 0.09C until 3.6V",
                "Hold at 3.6V until C/40",
                "Rest for 180 minutes",
            ),
            (
                "Charge at 0.18C until 4.05V",
                "Hold at 4.05V until C/25",
                "Discharge at 0.12C until 2.8V",
            ) * 2,
            (
                "Rest for 3 hours",
            ),
        ]),

        "slow_6": pybamm.Experiment([
            (
                "Rest for 75 minutes",
                "Charge at 0.055C until 3.95V",
                "Hold at 3.95V until C/45",
                "Charge at 0.04C until 4.1V",
                "Hold at 4.1V until C/40",
                "Discharge at 0.05C until 2.75V",
                "Rest for 1 hour",
            ),
        ]),

        "fast_6": pybamm.Experiment([
            (
                "Rest for 20 minutes",
                "Charge at 0.85C until 3.9V",
                "Hold at 3.9V until C/9",
                "Charge at 0.6C until 4.1V",
                "Hold at 4.1V until C/10",
                "Discharge at 0.6C until 2.75V",
                "Rest for 20 minutes",
            ),
        ]),
    }

    def safe_solution_array(solution, variable_name, t_uniform):
        values = solution[variable_name](t_uniform)
        values = np.asarray(values)
        if values.ndim == 2:
            values = np.mean(values, axis=0)
        return values

    for protocol, experiment in experiments_to_benchmark.items():
        print(f"\nRunning DFN simulation for {protocol} ...")

        sim = pybamm.Simulation(
            build_dfn_model(),
            experiment=experiment,
            parameter_values=parameter_values,
            solver=solver,
        )

        start = time.perf_counter()
        sol = sim.solve()
        solve_time = time.perf_counter() - start

        start = time.perf_counter()
        t_end = float(sol.t[-1])
        t_uniform = np.arange(0, t_end, 1.0)

        current = safe_solution_array(sol, "Current [A]", t_uniform)
        voltage = safe_solution_array(sol, "Terminal voltage [V]", t_uniform)
        temperature = safe_solution_array(sol, "Volume-averaged cell temperature [K]", t_uniform)
        sei_thickness_nm = safe_solution_array(sol, "X-averaged negative SEI thickness [m]", t_uniform) * 1e9
        lithium_capacity = safe_solution_array(sol, "Total lithium capacity [A.h]", t_uniform)

        sei_growth_rate_nm_per_s = np.diff(sei_thickness_nm, prepend=sei_thickness_nm[0])
        cumulative_charge_As = np.cumsum(np.abs(current)) * 1.0
        sqrt_cumulative_charge = np.sqrt(cumulative_charge_As)

        post_time = time.perf_counter() - start

        dfn_results[protocol] = {
            "protocol": protocol,
            "duration_h": t_end / 3600,
            "n_resampled_steps": len(t_uniform),
            "solve_time_s": solve_time,
            "postprocess_time_s": post_time,
            "solve_plus_postprocess_time_s": solve_time + post_time,
        }

        print("DFN timing:", dfn_results[protocol])

else:
    print("Skipping DFN timing because PyBaMM is unavailable.")


## Create summary table and speedup estimate

In [ ]:
rows = []

# Protocol-wise DFN and MLP timings
for protocol in protocols_to_benchmark:
    dfn_info = dfn_results.get(protocol)
    surrogate_info = surrogate_by_protocol.get(protocol)

    if dfn_info is not None:
        rows.append({
            "protocol": protocol,
            "method": "DFN simulation",
            "n_windows_or_steps": dfn_info["n_resampled_steps"],
            "runtime_mean_s": dfn_info["solve_plus_postprocess_time_s"],
            "runtime_std_s": np.nan,
            "duration_h": dfn_info["duration_h"],
            "speedup_vs_dfn": np.nan,
            "notes": "DFN solve + 1-second post-processing",
        })

    if surrogate_info is not None:
        if dfn_info is not None:
            speedup = dfn_info["solve_plus_postprocess_time_s"] / surrogate_info["mean_s"]
        else:
            speedup = np.nan

        rows.append({
            "protocol": protocol,
            "method": "MLP surrogate inference",
            "n_windows_or_steps": surrogate_info["n_windows"],
            "runtime_mean_s": surrogate_info["mean_s"],
            "runtime_std_s": surrogate_info["std_s"],
            "duration_h": dfn_info["duration_h"] if dfn_info is not None else np.nan,
            "speedup_vs_dfn": speedup,
            "notes": f"Mean over {surrogate_info['repeats']} repeated inference runs",
        })

# Timing for all test protocols together
rows.append({
    "protocol": "all_test",
    "method": "MLP surrogate inference",
    "n_windows_or_steps": surrogate_all["n_windows"],
    "runtime_mean_s": surrogate_all["mean_s"],
    "runtime_std_s": surrogate_all["std_s"],
    "duration_h": np.nan,
    "speedup_vs_dfn": np.nan,
    "notes": f"All non-overlapping test windows; mean over {surrogate_all['repeats']} repeated inference runs",
})

runtime_df = pd.DataFrame(rows)
runtime_df["runtime_mean_s"] = runtime_df["runtime_mean_s"].astype(float)

runtime_path = RESULTS_DIR / "runtime_comparison_dfn_vs_mlp_surrogate_protocolwise.csv"
runtime_df.to_csv(runtime_path, index=False)

print("Saved:", runtime_path.resolve())
display(runtime_df)

# Compact protocol-wise summary for manuscript table
summary_rows = []

for protocol in protocols_to_benchmark:
    dfn_row = runtime_df[(runtime_df["protocol"] == protocol) & (runtime_df["method"] == "DFN simulation")]
    mlp_row = runtime_df[(runtime_df["protocol"] == protocol) & (runtime_df["method"] == "MLP surrogate inference")]

    if len(dfn_row) > 0 and len(mlp_row) > 0:
        summary_rows.append({
            "Protocol": protocol,
            "DFN runtime (s)": float(dfn_row["runtime_mean_s"].iloc[0]),
            "MLP runtime (s)": float(mlp_row["runtime_mean_s"].iloc[0]),
            "Speedup": float(mlp_row["speedup_vs_dfn"].iloc[0]),
        })

summary_df = pd.DataFrame(summary_rows)

if len(summary_df) > 0:
    summary_path = RESULTS_DIR / "runtime_summary_for_paper.csv"
    summary_df.to_csv(summary_path, index=False)
    print("Saved:", summary_path.resolve())
    display(summary_df)

    print("\nLaTeX-friendly rounded summary:")
    rounded = summary_df.copy()
    rounded["DFN runtime (s)"] = rounded["DFN runtime (s)"].map(lambda x: f"{x:.2f}")
    rounded["MLP runtime (s)"] = rounded["MLP runtime (s)"].map(lambda x: f"{x:.4f}")
    rounded["Speedup"] = rounded["Speedup"].map(lambda x: f"{x:.1f}x")
    display(rounded)


## Manuscript sentence template

After running the notebook, use the generated `runtime_summary_for_paper.csv` values in a compact table or sentence.

```latex
Runtime benchmarking was performed on the multi-stage, slow, and fast test protocols. Across these protocols, DFN simulation required approximately X--Y s including post-processing, whereas the trained MLP surrogate generated the corresponding non-overlapping forecast windows in A--B s, corresponding to speedups of approximately C--D$\times$. This demonstrates the practical efficiency of the proposed surrogate for rapid protocol screening.
```
